# 02 - Limpieza de datos

## Objetivo

Realizar la limpieza de los datos a partir de los problemas identificados durante la etapa de comprensión, procurando conservar la información original y documentar las decisiones tomadas.

Los principales aspectos identificados que requieren revisión son:

- `TotalCharges` está almacenada como texto aunque conceptualmente es una variable numérica.
- Existen 11 registros con espacios en blanco en `TotalCharges`.
- Los 11 registros anteriores corresponden a clientes con `tenure = 0`.
- `customerID` corresponde a un identificador y no a una característica predictora.
- No se encontraron registros duplicados.
- Las variables categóricas no presentaron inconsistencias evidentes en sus valores.

## 1. Carga de los datos

In [37]:
from pathlib import Path
import pandas as pd

DATA_PATH = Path("../data/raw/Telco-Customer-Churn.csv")

df = pd.read_csv(DATA_PATH)

In [38]:
df_clean = df.copy()

## 2. Tratamiento de `TotalCharges`

Durante la etapa de comprensión de los datos se identificó que `TotalCharges` está almacenada como texto, aunque representa una variable numérica. Además, se encontraron 11 registros con espacios en blanco, todos correspondientes a clientes con `tenure = 0`.

Antes de realizar la conversión se revisarán nuevamente estos registros para definir el tratamiento más adecuado.

In [39]:
df_clean.loc[
    df_clean["TotalCharges"].str.strip().eq(""),
    ["customerID", "tenure", "MonthlyCharges", "TotalCharges"]
]

,customerID,tenure,MonthlyCharges,TotalCharges
488,4472-LVYGI,0,52.55,
753,3115-CZMZD,0,20.25,
936,5709-LVOEQ,0,80.85,
1082,4367-NUYAO,0,25.75,
1340,1371-DWPAZ,0,56.05,
3331,7644-OMVMY,0,19.85,
3826,3213-VVOLG,0,25.35,
4380,2520-SGTTA,0,20.00,
5218,2923-ARZLG,0,19.70,
6670,4075-WKNIU,0,73.35,


In [40]:
df_clean["TotalCharges"] = pd.to_numeric(
    df_clean["TotalCharges"],
    errors="coerce"
)

In [41]:
df_clean["TotalCharges"].isna().sum()


np.int64(11)

In [42]:
df_clean["TotalCharges"].dtype

dtype('float64')

In [43]:
df_clean.loc[
    df_clean["TotalCharges"].isna(),
    "TotalCharges"
] = 0

In [44]:
df_clean["TotalCharges"].isna().sum()


np.int64(0)

In [45]:
df_clean.loc[
    df_clean["tenure"] == 0,
    ["tenure", "MonthlyCharges", "TotalCharges"]
]

,tenure,MonthlyCharges,TotalCharges
488,0,52.55,0.0
753,0,20.25,0.0
936,0,80.85,0.0
1082,0,25.75,0.0
1340,0,56.05,0.0
3331,0,19.85,0.0
3826,0,25.35,0.0
4380,0,20.00,0.0
5218,0,19.70,0.0
6670,0,73.35,0.0


### Decisión tomada

Los 11 valores faltantes de `TotalCharges` correspondían a clientes con `tenure = 0`.

En lugar de eliminar estos registros o imputar una media o mediana, se asignó `0` a `TotalCharges`, ya que estos clientes no presentan meses completos de antigüedad y, por tanto, no tendrían cargos acumulados registrados.

Después de la corrección, `TotalCharges` quedó almacenada como una variable numérica de tipo `float64` y no presenta valores faltantes.

## 3. Verificación de duplicados

In [46]:
df_clean.duplicated().sum()

np.int64(0)

In [47]:
df_clean["customerID"].duplicated().sum()

np.int64(0)

## 4. Decisiones sobre variables

Durante la revisión se identificaron dos variables que requieren una consideración particular:

- `customerID` se conservará en el `dd_clean` para mantener la trazabilidad de los registros, pero será excluido posteriormente de las variables predictoras utilizadas para entrenar los modelos.
- `SeniorCitizen` es conceptualmente una variable categórica binaria, aunque está codificada numéricamente como `0` y `1`. Se conservará esta codificación, ya que representa adecuadamente sus dos categorías y no requiere una transformación durante esta etapa.

## 5. Verificación final

In [48]:
df_clean.isna().sum()

customerID          0
gender              0
SeniorCitizen       0
Partner             0
Dependents          0
tenure              0
PhoneService        0
MultipleLines       0
InternetService     0
OnlineSecurity      0
OnlineBackup        0
DeviceProtection    0
TechSupport         0
StreamingTV         0
StreamingMovies     0
Contract            0
PaperlessBilling    0
PaymentMethod       0
MonthlyCharges      0
TotalCharges        0
Churn               0
dtype: int64

In [49]:
df_clean.shape

(7043, 21)

In [50]:
df_clean.dtypes

customerID              str
gender                  str
SeniorCitizen         int64
Partner                 str
Dependents              str
tenure                int64
PhoneService            str
MultipleLines           str
InternetService         str
OnlineSecurity          str
OnlineBackup            str
DeviceProtection        str
TechSupport             str
StreamingTV             str
StreamingMovies         str
Contract                str
PaperlessBilling        str
PaymentMethod           str
MonthlyCharges      float64
TotalCharges        float64
Churn                   str
dtype: object

## 6. Exportación del dataset limpio

Una vez finalizada la limpieza y realizadas las verificaciones de calidad, se guarda el dataset resultante en `data/processed/`. De esta manera, las siguientes etapas del proyecto podrán trabajar con los datos limpios sin modificar el archivo original.

In [51]:
OUTPUT_PATH = Path("../data/processed/Telco-Customer-Churn-clean.csv")

OUTPUT_PATH.parent.mkdir(parents=True, exist_ok=True)

In [52]:
df_clean.to_csv(OUTPUT_PATH, index=False)

## 7. Resumen de la limpieza

Durante esta etapa se realizaron las siguientes acciones:

- Se convirtió `TotalCharges` de texto a tipo numérico.
- Se identificaron 11 valores no convertibles en `TotalCharges`, correspondientes a clientes con `tenure = 0`, y se asignó `0` como valor de cargos acumulados.
- Se verificó que no existen registros duplicados ni identificadores de cliente repetidos.
- Se conservaron los 7.043 registros y las 21 variables originales.
- `customerID` se mantuvo para conservar la trazabilidad de los registros, aunque será excluido posteriormente de las variables predictoras.
- `SeniorCitizen` se conservó con su codificación binaria original (`0` y `1`).
- Se verificó que el df_clean no contiene valores faltantes.
- El resultado final se almacenó en `data/processed/Telco-Customer-Churn-clean.csv`.

El dataset queda preparado para continuar con la etapa de análisis exploratorio.